[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-2-BACKUP_executed.ipynb)

---


# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 2: List Scheduling Algorithm

### Goal and workflow

This notebook implements a list-scheduling heuristic for a container yard. It repeatedly considers feasible moves of top containers, scores each candidate, and executes the candidate with the highest score. The aim is to arrange containers so that higher-priority containers are not blocked beneath lower-priority containers.

The inputs are a set of containers, their integer priorities, their stack and tier positions, and the bay capacity. A lower priority number means a higher retrieval priority. The outputs are the resulting container positions, the sequence of moves, and a log of the algorithm's progress.

### Model assumptions and scope

- Each stack has a fixed number of tiers. Tier 0 is the bottom position, and larger tier indices are higher positions.
- Containers in each stack occupy consecutive tiers starting at tier 0; gaps are not allowed.
- Only the top container of a stack can be moved. A move transfers it to the top of a different stack with available capacity.
- A priority violation is an adjacent pair in the same stack in which the lower container has a larger priority number than the container directly above it. The target arrangement has no such adjacent inversions.
- The score is a heuristic combining violation reduction, stack-height balance, move efficiency, and a priority bonus. Its weights express this example's scoring preferences; they are not physical units or a guarantee of optimality.
- The model does not represent travel times, equipment interactions, arrival uncertainty, or other operational constraints. The move limit bounds the number of iterations.

### Concrete example

The bay has three stacks. Listed from bottom to top, the initial configuration is:

- Stack 0: Container B (priority 2), then Container E (priority 5)
- Stack 1: Container D (priority 4), then Container A (priority 1)
- Stack 2: Container C (priority 3)

The algorithm can move only E, A, and C initially, because they are the top containers of their stacks. A move must also respect the target stack's capacity.


### Formal model of the heuristic

Let $C$ denote a bay configuration, let $h_s(C)$ be the height of stack $s$, and let $V(C)$ be the number of adjacent priority violations. For a candidate move $m$ from stack $i$ to stack $j$, let $C_m$ be the configuration after that move, let $k$ be the source tier of the moved container, and let $p_c$ be its priority. The mean stack height is

$$
\bar h = \frac{1}{S}\sum_{s=0}^{S-1} h_s(C),
$$

where $S$ is the number of stacks. The score used to rank a feasible move is

$$
\begin{aligned}
\operatorname{score}(m) ={}& 50\bigl(V(C)-V(C_m)\bigr) \\
&+20\Bigl(\lvert h_i(C)-\bar h\rvert+\lvert h_j(C)-\bar h\rvert \\
&\qquad\quad-\lvert h_i(C)-1-\bar h\rvert-\lvert h_j(C)+1-\bar h\rvert\Bigr) \\
&+\max\bigl(0,15-2\lvert i-j\rvert-k\bigr) \\
&+15\frac{10-p_c}{10}.
\end{aligned}
$$

The first term is the exact change in the total number of violations, so it can be negative when a move creates violations. The second term rewards a reduction in the combined deviation of the source and target stack heights from the current mean. The third term favors smaller horizontal stack distances and lower source-tier indices. The final term gives a larger bonus to containers with smaller priority numbers; it uses a priority scale anchored at 10.

At each iteration, the algorithm generates all feasible top-container moves, evaluates their scores, and selects a move with the maximum score. If several moves tie, Python's `max` selects the first one in the generated order. The stopping conditions are zero violations, no feasible moves, or reaching the move limit. This greedy rule does not guarantee that every move improves the configuration or that the final arrangement uses the fewest possible moves.

If there are $S$ stacks, $T$ tiers, $N$ containers, and a move limit $M$, there are at most $S(S-1)$ candidate moves per iteration. In this direct implementation, scoring each candidate scans container and stack-position data, giving a polynomial upper bound of $O\bigl(MS^3(N+T)\bigr)$ time and $O(N+ST+S^2)$ working storage. The actual number of iterations can be smaller than $M$ because the algorithm may stop earlier.


In [ ]:
import time
from dataclasses import dataclass
from typing import List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

plt.style.use('default')

In [ ]:
@dataclass
class Container:
    """A container's priority and current bay position."""

    id: str
    priority: int  # Lower values indicate higher retrieval priority.
    current_stack: int
    current_tier: int


@dataclass
class Move:
    """A transfer of one top container between stacks."""

    container_id: str
    from_stack: int
    from_tier: int
    to_stack: int
    to_tier: int
    score: float


@dataclass
class ListScheduler:
    """Greedy list scheduler for the yard pre-marshalling problem."""

    stacks: int
    tiers: int

    def __init__(self, stacks: int, tiers: int):
        if stacks <= 0 or tiers <= 0:
            raise ValueError('The number of stacks and tiers must be positive.')

        self.stacks = stacks
        self.tiers = tiers
        self.move_history: List[Move] = []
        self.execution_log: List[str] = []

    def validate_configuration(self, containers: List[Container]) -> None:
        """Check that a bay state has unique containers and compact stacks."""
        container_ids = [container.id for container in containers]
        if len(container_ids) != len(set(container_ids)):
            raise ValueError('Container IDs must be unique.')

        occupied_positions = set()
        for container in containers:
            if not 0 <= container.current_stack < self.stacks:
                raise ValueError(f'Container {container.id} has an invalid stack.')
            if not 0 <= container.current_tier < self.tiers:
                raise ValueError(f'Container {container.id} has an invalid tier.')

            position = (container.current_stack, container.current_tier)
            if position in occupied_positions:
                raise ValueError('A stack position cannot contain multiple containers.')
            occupied_positions.add(position)

        for stack in range(self.stacks):
            stack_tiers = sorted(
                container.current_tier
                for container in containers
                if container.current_stack == stack
            )
            if stack_tiers != list(range(len(stack_tiers))):
                raise ValueError('Containers in each stack must occupy consecutive tiers from tier 0.')

    def get_stack_height(self, containers: List[Container], stack: int) -> int:
        """Return the number of containers in a stack."""
        return sum(container.current_stack == stack for container in containers)

    def get_top_container(
        self, containers: List[Container], stack: int
    ) -> Optional[Container]:
        """Return the highest-tier container, or None for an empty stack."""
        stack_containers = [
            container
            for container in containers
            if container.current_stack == stack
        ]
        if not stack_containers:
            return None
        return max(stack_containers, key=lambda container: container.current_tier)

    def count_priority_violations(self, containers: List[Container]) -> int:
        """Count adjacent pairs with a lower-priority container underneath."""
        grid = [[None for _ in range(self.tiers)] for _ in range(self.stacks)]
        for container in containers:
            grid[container.current_stack][container.current_tier] = container.priority

        violations = 0
        for stack in range(self.stacks):
            for tier in range(self.tiers - 1):
                lower_priority = grid[stack][tier]
                upper_priority = grid[stack][tier + 1]
                if (
                    lower_priority is not None
                    and upper_priority is not None
                    and lower_priority > upper_priority
                ):
                    violations += 1
        return violations

    def calculate_move_score(
        self, containers: List[Container], move: Move
    ) -> float:
        """Score a feasible move using the four model components."""
        container = next(
            item for item in containers if item.id == move.container_id
        )

        # Evaluate the actual resulting configuration so the violation term
        # reflects every affected adjacent pair, not an assumed improvement.
        moved_containers = self.execute_move(containers, move)
        violation_reduction = (
            self.count_priority_violations(containers)
            - self.count_priority_violations(moved_containers)
        )
        score = 50 * violation_reduction

        from_height = self.get_stack_height(containers, move.from_stack)
        to_height = self.get_stack_height(containers, move.to_stack)
        average_height = sum(
            self.get_stack_height(containers, stack)
            for stack in range(self.stacks)
        ) / self.stacks

        balance_before = (
            abs(from_height - average_height)
            + abs(to_height - average_height)
        )
        balance_after = (
            abs(from_height - 1 - average_height)
            + abs(to_height + 1 - average_height)
        )
        score += 20 * (balance_before - balance_after)

        distance = abs(move.from_stack - move.to_stack)
        efficiency = max(0, 15 - 2 * distance - move.from_tier)
        score += efficiency

        priority_bonus = 15 * (10 - container.priority) / 10
        score += priority_bonus
        return score

    def generate_valid_moves(self, containers: List[Container]) -> List[Move]:
        """Generate and score all capacity-feasible top-container moves."""
        moves = []
        for stack in range(self.stacks):
            top_container = self.get_top_container(containers, stack)
            if top_container is None:
                continue

            for target_stack in range(self.stacks):
                if target_stack == stack:
                    continue

                target_height = self.get_stack_height(containers, target_stack)
                if target_height >= self.tiers:
                    continue

                move = Move(
                    container_id=top_container.id,
                    from_stack=stack,
                    from_tier=top_container.current_tier,
                    to_stack=target_stack,
                    to_tier=target_height,
                    score=0.0,
                )
                move.score = self.calculate_move_score(containers, move)
                moves.append(move)
        return moves

    def execute_move(
        self, containers: List[Container], move: Move
    ) -> List[Container]:
        """Apply a legal move to a copied state and return that state."""
        self.validate_configuration(containers)
        if move.from_stack == move.to_stack:
            raise ValueError('A move must transfer a container to a different stack.')
        if not 0 <= move.to_stack < self.stacks:
            raise ValueError('The target stack is outside the bay.')

        top_container = self.get_top_container(containers, move.from_stack)
        if top_container is None or top_container.id != move.container_id:
            raise ValueError('Only the top container of the source stack can be moved.')

        target_height = self.get_stack_height(containers, move.to_stack)
        if target_height >= self.tiers or move.to_tier != target_height:
            raise ValueError('The target position must be the available top tier.')

        new_containers = [
            Container(
                container.id,
                container.priority,
                container.current_stack,
                container.current_tier,
            )
            for container in containers
        ]
        moved_container = next(
            container
            for container in new_containers
            if container.id == move.container_id
        )
        moved_container.current_stack = move.to_stack
        moved_container.current_tier = move.to_tier
        return new_containers

    def solve(
        self, initial_containers: List[Container], max_moves: int = 100
    ) -> Tuple[List[Container], List[Move]]:
        """Run greedy list scheduling until a stopping condition is met."""
        if max_moves < 0:
            raise ValueError('The move limit cannot be negative.')
        self.validate_configuration(initial_containers)

        containers = [
            Container(
                container.id,
                container.priority,
                container.current_stack,
                container.current_tier,
            )
            for container in initial_containers
        ]
        self.move_history = []
        self.execution_log = []

        iteration = 0
        while iteration < max_moves:
            violations = self.count_priority_violations(containers)
            if violations == 0:
                self.execution_log.append(
                    f'Iteration {iteration}: No adjacent priority violations remain.'
                )
                break

            valid_moves = self.generate_valid_moves(containers)
            if not valid_moves:
                self.execution_log.append(
                    f'Iteration {iteration}: No feasible moves are available.'
                )
                break

            best_move = max(valid_moves, key=lambda candidate: candidate.score)
            containers = self.execute_move(containers, best_move)
            self.move_history.append(best_move)

            new_violations = self.count_priority_violations(containers)
            self.execution_log.append(
                f'Iteration {iteration}: Move {best_move.container_id} from '
                f'stack {best_move.from_stack} to stack {best_move.to_stack}, '
                f'score={best_move.score:.1f}, violations: '
                f'{violations} -> {new_violations}'
            )
            iteration += 1

        if (
            iteration == max_moves
            and self.count_priority_violations(containers) > 0
        ):
            self.execution_log.append(
                f'Move limit reached after {max_moves} moves.'
            )

        return containers, self.move_history

### Reading the implementation

`Container` stores each container's priority and position; `Move` records a proposed transfer and its score. `validate_configuration` checks that positions are unique, within the bay, and compact from the bottom of each stack. These properties make the top-container and target-tier rules well-defined.

`count_priority_violations` evaluates the objective measure $V(C)$. `generate_valid_moves` enumerates the currently accessible containers and capacity-feasible destination stacks. `calculate_move_score` evaluates the four terms in the model, while `execute_move` applies a move to a separate copy of the state. Copying prevents the evaluation of candidate moves from changing the configuration being considered. Finally, `solve` repeats candidate generation, scoring, and greedy selection in that order.


In [ ]:
def visualize_bay_state(
    containers: List[Container],
    stacks: int,
    tiers: int,
    title: str,
    ax=None,
):
    """Draw stack positions and encode priority with a labeled color scale."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 6))

    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        grid[container.current_stack][container.current_tier] = container

    priorities = [container.priority for container in containers]
    if priorities:
        minimum_priority = min(priorities)
        maximum_priority = max(priorities)
        if minimum_priority == maximum_priority:
            minimum_priority -= 0.5
            maximum_priority += 0.5
    else:
        minimum_priority, maximum_priority = 0, 1

    color_scale = Normalize(vmin=minimum_priority, vmax=maximum_priority)
    colormap = plt.cm.RdYlBu_r

    for stack in range(stacks):
        for tier in range(tiers):
            container = grid[stack][tier]
            x_position = stack * 1.2
            y_position = tier * 0.8
            if container is None:
                rectangle = plt.Rectangle(
                    (x_position, y_position),
                    1.0,
                    0.6,
                    facecolor='lightgray',
                    edgecolor='gray',
                    linewidth=1,
                    alpha=0.3,
                )
            else:
                rectangle = plt.Rectangle(
                    (x_position, y_position),
                    1.0,
                    0.6,
                    facecolor=colormap(color_scale(container.priority)),
                    edgecolor='black',
                    linewidth=2,
                )
                ax.text(
                    x_position + 0.5,
                    y_position + 0.3,
                    f'{container.id}\n(Priority {container.priority})',
                    ha='center',
                    va='center',
                    fontweight='bold',
                    fontsize=9,
                )
            ax.add_patch(rectangle)

    ax.set_xlim(-0.5, stacks * 1.2)
    ax.set_ylim(-0.5, tiers * 0.8)
    ax.set_xlabel('Stack index')
    ax.set_ylabel('Tier index (0 = bottom)')
    ax.set_title(title, fontweight='bold')
    ax.set_xticks([stack * 1.2 + 0.5 for stack in range(stacks)])
    ax.set_xticklabels([f'S{stack}' for stack in range(stacks)])
    ax.set_yticks([tier * 0.8 + 0.3 for tier in range(tiers)])
    ax.set_yticklabels([f'T{tier}' for tier in range(tiers)])
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

    color_map = plt.cm.ScalarMappable(norm=color_scale, cmap=colormap)
    ax.figure.colorbar(
        color_map,
        ax=ax,
        label='Priority (lower values indicate higher priority)',
    )


def visualize_algorithm_progression(
    initial_containers: List[Container],
    final_containers: List[Container],
    move_history: List[Move],
    stacks: int,
    tiers: int,
):
    """Show the initial state and selected states from the move sequence."""
    number_of_moves = len(move_history)
    if number_of_moves == 0:
        return

    number_of_stages = min(5, number_of_moves + 1)
    figure, axes = plt.subplots(
        1, number_of_stages, figsize=(5 * number_of_stages, 6)
    )
    if number_of_stages == 1:
        axes = [axes]

    visualize_bay_state(
        initial_containers, stacks, tiers, 'Initial state', axes[0]
    )

    for stage_index in range(1, number_of_stages):
        move_index = round(
            stage_index * number_of_moves / (number_of_stages - 1)
        ) - 1
        if stage_index == number_of_stages - 1:
            stage_containers = [
                Container(
                    container.id,
                    container.priority,
                    container.current_stack,
                    container.current_tier,
                )
                for container in final_containers
            ]
            title = f'Final state ({number_of_moves} moves)'
        else:
            stage_containers = [
                Container(
                    container.id,
                    container.priority,
                    container.current_stack,
                    container.current_tier,
                )
                for container in initial_containers
            ]
            for move in move_history[: move_index + 1]:
                moved_container = next(
                    container
                    for container in stage_containers
                    if container.id == move.container_id
                )
                moved_container.current_stack = move.to_stack
                moved_container.current_tier = move.to_tier
            title = f'After {move_index + 1} moves'

        visualize_bay_state(
            stage_containers, stacks, tiers, title, axes[stage_index]
        )

    figure.tight_layout()
    plt.show()

### Create and inspect the example bay

The tier coordinates below make the bottom-to-top order explicit. The bay has four available tiers per stack, so the example has spare capacity for transfers.

In [ ]:
print('=' * 60)
print('YARD PRE-MARSHALLING: LIST SCHEDULING')
print('=' * 60)

containers = [
    Container('E', 5, 0, 1),
    Container('B', 2, 0, 0),
    Container('A', 1, 1, 1),
    Container('D', 4, 1, 0),
    Container('C', 3, 2, 0),
]

stacks, tiers = 3, 4
scheduler = ListScheduler(stacks, tiers)
scheduler.validate_configuration(containers)

print('\nBay configuration:')
print(f'  Stacks: {stacks}')
print(f'  Tiers per stack: {tiers}')
print(f'  Containers: {len(containers)}')

print('\nInitial container positions:')
for container in containers:
    print(
        f'  Container {container.id}: priority {container.priority}, '
        f'stack {container.current_stack}, tier {container.current_tier}'
    )

initial_violations = scheduler.count_priority_violations(containers)
print(f'\nInitial adjacent priority violations: {initial_violations}')

visualize_bay_state(
    containers, stacks, tiers, 'Initial bay configuration'
)

### Run the list-scheduling heuristic

The elapsed time measures this run in the current environment; it is not a hardware-independent performance guarantee. The execution log records the selected move, its score, and the violation count before and after that move.

In [ ]:
print('\n' + '=' * 60)
print('LIST-SCHEDULING EXECUTION')
print('=' * 60)

start_time = time.time()
final_containers, move_history = scheduler.solve(containers, max_moves=50)
execution_time = time.time() - start_time

print(f'\nElapsed time: {execution_time:.4f} seconds')
print(f'Moves executed: {len(move_history)}')

print('\nExecution log:')
for log_entry in scheduler.execution_log:
    print(f'  {log_entry}')

### Interpret the resulting arrangement

A zero violation count means that every adjacent pair in each stack satisfies the modeled priority order. It does not establish that the move sequence is shortest or that the resulting stack heights are perfectly balanced. If violations remain, inspect the stopping log to distinguish the move limit from the absence of feasible moves.

In [ ]:
print('\n' + '=' * 60)
print('FINAL SOLUTION ANALYSIS')
print('=' * 60)

print('\nFinal container positions:')
for container in final_containers:
    print(
        f'  Container {container.id}: priority {container.priority}, '
        f'stack {container.current_stack}, tier {container.current_tier}'
    )

scheduler.validate_configuration(final_containers)
final_violations = scheduler.count_priority_violations(final_containers)

print('\nSolution summary:')
print(f'  Initial adjacent priority violations: {initial_violations}')
print(f'  Final adjacent priority violations: {final_violations}')
print(f'  Moves executed: {len(move_history)}')

if final_violations == 0:
    print('  The final configuration has no adjacent priority violations.')
else:
    print(f'  The final configuration retains {final_violations} violations.')

visualize_algorithm_progression(
    containers,
    final_containers,
    move_history,
    stacks,
    tiers,
)

### Strengths, limitations, and appropriate interpretation

The method makes its decisions through an explicit weighted score and considers only currently accessible containers and capacity-feasible destinations. Its repeated candidate evaluation is polynomial in the bay dimensions and the chosen move limit, so the computation is bounded for a fixed instance and limit.

The score is a heuristic rather than an optimization proof. A high-scoring move can create additional violations, and a greedy sequence can reach the move limit without removing every violation. The weights also influence which trade-offs the scheduler prefers. Therefore, interpret the output as the result of this specific scoring rule and stopping policy, not as a guaranteed minimum-move or globally optimal plan.

The model is useful for studying how accessibility, stack capacity, and competing move preferences interact. Applying it to an operational yard would require representing any additional constraints and costs that matter to that setting.